# Fine-tuning ModernBERT with LoRA: Signpost

**Part 4 of "LLMs the Hard Way"** (branch `signpost`). The frozen encoder of `02` plateaued at 114 / 170 on grounded questions; here ModernBERT-large learns through **LoRA** adapters (7.2 M trained parameters, 395 M frozen) plus a scoring head. One model, one format: **agent routing** and **grounded questions** (yes / no, choice, score).

```text
<role>: <msg> [SEP] … [SEP] current agent: X [SEP] question: <q> [SEP] candidate: <option>. <description>
        -> ModernBERT-large (+ LoRA) -> mean pool -> head -> score;   softmax over the options -> answer
```

The published [skamalj/signpost](https://huggingface.co/skamalj/signpost) is run 25 of this recipe (tag `signpost-v1.0`). A rerun is compared with it; nothing is uploaded.

| Test (never trained on) | Contents | Items |
|---|---|---|
| E10 | Realistic chats, 13 companies, routed live | 193 turns; 83 at 6 test companies |
| E11 | One opening message, 5 agents | 100 |
| Banking | Banking conversations, 5 agents | 50 · 225 |
| E12 | Grounded questions | 170 (58 yes / no, 59 choice, 53 score) |
| Independent | Outside author: questions, routing, first messages | 300 · 172 turns · 200 |
| Independent 2 | Second outside set, up to 50 options | 200 · 70 turns · 50 |

## Parameters

Run 25's recipe. Values read with `env` can be overridden by environment variables; `LORA_SMOKE=1` is a tiny check on ModernBERT-base.

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

SMOKE = env("LORA_SMOKE", "0") == "1"
ENCODER = "answerdotai/ModernBERT-base" if SMOKE else "answerdotai/ModernBERT-large"
# LoRA: rank, scale (the update is multiplied by alpha / r), dropout, target layers
LORA_R = int(env("LORA_R", "16"))
LORA_ALPHA = float(env("LORA_ALPHA", "32"))
LORA_DROPOUT = 0.1
LORA_TARGETS = ["Wqkv", "Wo", "Wi"]           # attention (Wqkv, Wo) and MLP input (Wi)
HEAD_HIDDEN, HEAD_DROPOUT = 256, 0.1          # head: Linear(H, 256) -> GELU -> Linear(256, 1)
# Optimisation
LR = float(env("LR", "2e-4"))                 # LoRA adapters
HEAD_LR = float(env("HEAD_LR", "1e-3"))       # head
WEIGHT_DECAY, WARMUP_FRAC, CLIP_NORM = 0.01, 0.06, 1.0
EPOCHS = int(env("EPOCHS", "1" if SMOKE else "2"))
BATCH_ROUTE = int(env("BATCH_ROUTE", "4"))    # routing questions per step
BATCH_CHOICE = int(env("BATCH_CHOICE", "16"))  # grounded questions per step
MAX_LEN = int(env("MAX_LEN", "320"))          # tokens per input; the oldest history is cut first
LOG_EVERY = 100                               # steps between progress lines
# Score questions: "ordinal" (order-aware) or "plain" cross-entropy
SCORE_LOSS = env("SCORE_LOSS", "ordinal")
SCORE_EPS = 0.1                               # share of the target given to neighbouring levels
SCORE_DIST_W = 0.5                            # weight of |expected level - true level|
# Seeds and the collapse guard: minimum dev accuracy after epoch 1, else restart at seed + 1000
SEED = int(env("SEED", "42"))
SEED_STEP, ATTEMPTS = 1000, 3
COLLAPSE_DEV = {"routing": 0.95, "questions": 0.85}   # healthy runs had >= 0.977 / 0.949
SAVE_MODEL = env("SAVE_MODEL", "1") == "1"
# Input format
SEP = " [SEP] "
ROUTE_Q = "Which agent should handle the user's latest message?"

## Setup

Pinned packages, imports, the device, and `log` (prints and appends to `lora_signpost_run.log`).

In [2]:
# Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip uninstall -y -q torchao
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, random, shutil, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"                   # fp16 mixed precision on a GPU
RUN_LOG = Path("lora_signpost_run.log")

def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")

def gpu():
    if not AMP:
        return "CPU"
    used = torch.cuda.memory_allocated(DEVICE) / 1e9
    peak = torch.cuda.max_memory_allocated(DEVICE) / 1e9
    return f"GPU {used:.1f} GB (peak {peak:.1f})"

def recent(losses):                         # mean of the last 50 losses, for progress lines
    return statistics.mean(losses[-50:] or [0])

versions = (f"torch {torch.__version__} | transformers {transformers.__version__}"
            f" | peft {peft.__version__}")
device_name = f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else ""
log(f"{versions} | {DEVICE}" + device_name + f" | {ENCODER}" + (" | SMOKE CHECK" if SMOKE else ""))

[06:16:42] torch 2.11.0+cu130 | transformers 5.17.0 | peft 0.21.2 | cuda:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition) | answerdotai/ModernBERT-large


---
# Data

The data is read from this repository (cloned first on Kaggle / Colab). Every training item, routing or question, becomes `{task, messages, current, question, options {name: description}, answer}`.

In [3]:
DATA_BRANCH = "signpost"
REPO_URL = "https://github.com/skamalj/llms-the-hard-way.git"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():                     # on Kaggle / Colab: clone, or refresh a clone
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH, REPO_URL,
                        str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH],
                       check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"],
                       check=True)
    DATA_DIR = repo / "data" / "modernbert_router"

def load(path):
    return json.load(open(DATA_DIR / path))

domains = {**load("generic/domains.json"), **load("realistic_train/domains.json")}

def route_item(ex):
    """A training conversation; its domain's 5 agents are the options."""
    return {"task": "routing", "domain": ex["domain"], "messages": ex["messages"],
            "current": ex["current_agent"], "question": ROUTE_Q,
            "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]}

def choice_item(it):
    """A grounded question: the text is the user's message; options have no description."""
    return {"task": it["type"], "domain": it["domain"],
            "messages": [{"role": "user", "content": it["state"]}], "current": None,
            "question": it["question"], "options": {o: "" for o in it["options"]},
            "answer": it["answer"], "state": it["state"],
            "levels": list(it["options"])}    # score questions: the levels, low to high

route_convs = load("generic/conversations.json") + load("realistic_train/conversations.json")
route_items = [route_item(ex) for ex in route_convs]
choice_items = [choice_item(it) for it in load("choice_train/items.json")["items"]]

Dev sets (best epoch and collapse guard only): 10% of the routing **domains** and 10% of the question **texts**, drawn with a seeded generator. **Expect** the training and dev sizes.

In [4]:
rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items})
rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items})
rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])

train_route = [x for x in route_items if x["domain"] not in dev_domains]
dev_route = [x for x in route_items if x["domain"] in dev_domains]
train_choice = [x for x in choice_items if x["state"] not in dev_texts]
dev_choice = [x for x in choice_items if x["state"] in dev_texts]
if SMOKE:
    train_route, dev_route = train_route[::400], dev_route[::100]
    train_choice, dev_choice = train_choice[::400], dev_choice[::100]

splits = [("train routing", train_route), ("train questions", train_choice),
          ("dev routing", dev_route), ("dev questions", dev_choice)]
for name, xs in splits:
    by_task = dict(Counter(x["task"] for x in xs))
    print(f"{name:15s}: {len(xs):5d} | {by_task}")
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")

def make_batches():
    """Routing batches of BATCH_ROUTE, then question batches of BATCH_CHOICE."""
    batches = [train_route[i:i + BATCH_ROUTE] for i in range(0, len(train_route), BATCH_ROUTE)]
    batches += [train_choice[i:i + BATCH_CHOICE]
                for i in range(0, len(train_choice), BATCH_CHOICE)]
    return batches

train routing  :  6319 | {'routing': 6319}
train questions:  9586 | {'noul': 5070, 'choice': 2596, 'score': 1920}
dev routing    :   531 | {'routing': 531}
dev questions  :  1014 | {'noul': 550, 'choice': 236, 'score': 228}
dev routing domains (unseen in training): ['campsite', 'hearing_clinic', 'home_cinema_installer', 'hr', 'motorbike_rental', 'museum', 'opticians']


The test sets, the banking agents (as in `02`), and E10's 6 test companies. **Expect** the size of every test.

In [5]:
real_convs = load("realistic/conversations.json")                      # E10
ft = load("first_turn/messages.json")                                  # E11, with its agents
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q, ind_r, ind_f = (load("independent/questions.json")["items"],
                       load("independent/routing.json"),
                       load("independent/first_turn.json")["messages"])
ind2_q, ind2_r, ind2_f = (load("independent2/questions.json")["items"],
                          load("independent2/routing.json"),
                          load("independent2/first_turn.json")["messages"])
BANKING_AGENTS = {                      # exactly as in 02_frozen_router.ipynb, Cell 9
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, "
        "payment charges and duplicate charges, refunds for transactions, and settlement "
        "problems where money was sent but not received. Does not handle card servicing "
        "such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and "
        "unblocking, lost or stolen cards, card replacement and delivery, card limits, card PIN, "
        "card security and compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, "
        "investment transactions, redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and "
        "policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact "
        "details, issues that do not clearly belong to a specialist, and routing customers to "
        "the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])         # the 6 test companies of 02, Cell 16
if SMOKE:
    real_convs, e11, banking, e12 = real_convs[:2], e11[::20], banking[:6], e12[::20]
    ind_q, ind_r, ind_f = ind_q[::30], ind_r[:2], ind_f[::20]
    ind2_q, ind2_r, ind2_f = ind2_q[::20], ind2_r[:2], ind2_f[::10]

def turns(convs):
    return sum(len(c["user_turns"]) for c in convs)

print(f"E10 {len(real_convs)} conversations / {turns(real_convs)} turns | E11 {len(e11)}"
      f" | banking {len(banking)} | E12 {len(e12)}")
print(f"independent: questions {len(ind_q)} | routing {len(ind_r)} conversations"
      f" / {turns(ind_r)} turns | first messages {len(ind_f)}")
print(f"independent 2: questions {len(ind2_q)} (up to {max(len(q['options']) for q in ind2_q)}"
      f" options) | routing {len(ind2_r)} conversations / {turns(ind2_r)} turns"
      f" (up to {max(len(c['agents']) for c in ind2_r)} agents) | first messages {len(ind2_f)}")

E10 35 conversations / 193 turns | E11 100 | banking 225 | E12 170
independent: questions 300 | routing 40 conversations / 172 turns | first messages 200
independent 2: questions 200 (up to 50 options) | routing 12 conversations / 70 turns (up to 11 agents) | first messages 50


---
# The model

`input_text` writes one (item, option) pair in `signpost.py`'s format. `Model` is ModernBERT with `peft` LoRA (`y = W x + (α / r)·B(A x)`) plus the head; `scores` scores every option of every item in one pass, as `[items, most options]`.

In [6]:
def input_text(item, name):
    desc = item["options"][name]
    parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
    parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"]
    parts.append(f"candidate: {name}. {desc}" if desc else f"candidate: {name}")
    return SEP.join(parts)

class Model:
    def __init__(self):                     # order fixed: LoRA and head weights use the seeded RNG
        self.tok = AutoTokenizer.from_pretrained(ENCODER)
        self.tok.padding_side, self.tok.truncation_side = "right", "left"   # cut oldest history
        enc = AutoModel.from_pretrained(ENCODER, dtype=torch.float32).to(DEVICE)
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
                                                  lora_dropout=LORA_DROPOUT,
                                                  target_modules=LORA_TARGETS))
        self.head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(self.H, HEAD_HIDDEN),
                                  nn.GELU(), nn.Dropout(HEAD_DROPOUT),
                                  nn.Linear(HEAD_HIDDEN, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode)
        self.head.train(mode)

    def scores(self, items):
        sizes = torch.tensor([len(it["options"]) for it in items], device=DEVICE)
        texts = [input_text(it, name) for it in items for name in it["options"]]
        enc = self.tok(texts, padding=True, truncation=True, max_length=MAX_LEN,
                       return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            out = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            hidden = out.last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)   # mean pool -> score
        real = torch.arange(int(sizes.max()), device=DEVICE) < sizes[:, None]   # slots in use
        grid = torch.full(real.shape, -1e4, device=DEVICE)
        grid[real] = s                      # options in order, item by item
        return grid

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

---
# Training

The loss is listwise cross-entropy over each question's options. For score questions, the ordinal variant moves 10% of the target to the neighbouring levels and adds 0.5 × |expected level − true level| / (levels − 1).

In [7]:
def scale_positions(items, K, scores):
    """[B, K] position of each slot on its scale; -9 for non-score items and empty slots."""
    rows = []
    for it in items:
        if it["task"] == "score":
            positions = [it["levels"].index(n) for n in it["options"]]
            rows.append(positions + [-9] * (K - len(it["options"])))
        else:
            rows.append([-9] * K)
    return torch.tensor(rows, device=scores.device, dtype=scores.dtype)

def soft_target(level, true, target, is_score, K):
    nbr = ((level - true).abs() == 1) & is_score[:, None]       # neighbouring levels
    n_nbr = nbr.sum(1, keepdim=True)
    eps = SCORE_EPS * (n_nbr > 0)
    return torch.where(nbr, eps / n_nbr.clamp(min=1), F.one_hot(target, K) * (1 - eps))

def level_distance(logp, level, true, items, is_score, scores):
    n_levels = torch.tensor([len(it["options"]) for it in items],
                            device=scores.device, dtype=scores.dtype)
    expected = (logp.exp() * level.clamp(min=0)).sum(1)
    distance = (expected - true[:, 0]).abs() / (n_levels - 1).clamp(min=1)
    return distance[is_score].mean()

def ordinal_loss(scores, items, target):
    logp = F.log_softmax(scores, dim=1)
    K = scores.shape[1]
    is_score = torch.tensor([it["task"] == "score" for it in items], device=scores.device)
    level = scale_positions(items, K, scores)
    true = level.gather(1, target[:, None])              # [B, 1]
    soft = soft_target(level, true, target, is_score, K)
    loss = -(soft * logp).sum(1).mean()
    if not is_score.any():
        return loss
    return loss + SCORE_DIST_W * level_distance(logp, level, true, items, is_score, scores)

One training step (options shuffled so position can't be learned) and the dev check: below `COLLAPSE_DEV` after epoch 1 raises `Collapsed`.

In [8]:
def dev_accuracy(model, items, batch=16):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            picks = model.scores(chunk).argmax(1).tolist()
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, picks))
    return ok / max(1, len(items)), ok

class Collapsed(Exception):
    """Training collapsed; retry with another seed."""

def end_of_epoch(model, epoch):
    acc_r, ok_r = dev_accuracy(model, dev_route)
    acc_c, ok_c = dev_accuracy(model, dev_choice)
    msg = (f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%}) | "
           f"dev questions {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
    log(f"epoch {epoch} done | {msg}")
    if epoch == 1 and (acc_r < COLLAPSE_DEV["routing"] or acc_c < COLLAPSE_DEV["questions"]):
        raise Collapsed(f"{msg} after epoch 1 (thresholds {COLLAPSE_DEV})")
    return (acc_r + acc_c) / 2

def train_step(model, opt, sched, scaler, params, batch):
    items = [dict(it) for it in batch]
    for it in items:                        # options shuffled, so position can't be learned
        names = list(it["options"])
        random.shuffle(names)
        it["options"] = {n: it["options"][n] for n in names}
    target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
    out = model.scores(items)
    if SCORE_LOSS == "ordinal":
        loss = ordinal_loss(out, items, target)
    else:
        loss = F.cross_entropy(out, target)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, CLIP_NORM)
    scaler.step(opt)
    scaler.update()
    sched.step()
    return loss

The training loop: AdamW with separate rates for adapters and head, 6% warm-up then linear decay, batches shuffled each epoch. The epoch with the best mean dev accuracy is restored at the end.

In [9]:
def train_model(model, seed):
    batches = make_batches()
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR},
                             {"params": model.head.parameters(), "lr": HEAD_LR}],
                            weight_decay=WEIGHT_DECAY)
    total = EPOCHS * len(batches)
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best_acc, best, best_epoch, step, t0 = -1.0, None, 0, 0, time.perf_counter()
    log(f"training: {len(batches)} batches per epoch ({len(train_route)} routing + "
        f"{len(train_choice)} questions), {total} steps, score loss {SCORE_LOSS}")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        run = {"routing": [], "questions": []}
        for bi in order:
            loss = train_step(model, opt, sched, scaler, params, batches[bi])
            kind = "routing" if batches[bi][0]["task"] == "routing" else "questions"
            run[kind].append(loss.item())
            step += 1
            if step % LOG_EVERY == 0 or step == total:
                rate = step / (time.perf_counter() - t0)
                log(f"  epoch {epoch} step {step}/{total} | "
                    f"loss routing {recent(run['routing']):.4f} "
                    f"questions {recent(run['questions']):.4f} | {rate:.2f} steps/s | "
                    f"~{(total - step) / rate / 60:.1f} min left | {gpu()}")
        mean_acc = end_of_epoch(model, epoch)
        if mean_acc > best_acc:             # keep copies of the best epoch's LoRA weights and head
            best_acc, best_epoch = mean_acc, epoch
            lora = model.enc.named_parameters()
            best = ({n: p.detach().clone() for n, p in lora if p.requires_grad},
                    copy.deepcopy(model.head.state_dict()))
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[0].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best_epoch} "
        f"(mean dev accuracy {best_acc:.1%})")
    return {"train_minutes": minutes, "best_epoch": best_epoch}

---
# Tests

Each test question is scored alone and timed with the GPU synchronized, tokenization included. `table` prints right / total per group and for all rows.

In [10]:
def lat_summary(ms):
    ms = sorted(ms)
    p95 = ms[min(len(ms) - 1, int(0.95 * len(ms)))]
    return {"p50": ms[len(ms) // 2], "p95": p95, "n": len(ms)}

def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000

def table(rows, key, cols, groups=None):
    """cols = {column: row -> True if right}; groups default to the sorted keys."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g)) for g in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g in groups:
        rs = [r for r in rows if g == "ALL" or key(r) == g]
        if rs:
            cells = " | ".join(f"{sum(map(f, rs)):>5} / {len(rs):<4}" for f in cols.values())
            print(f"  {g:{w}s} | {len(rs):>4} | " + cells)

def user(text):
    return [{"role": "user", "content": text}]

def routing_item(messages, current, agents):
    return {"messages": messages, "current": current, "question": ROUTE_Q, "options": agents}

The four kinds of test. In live routing the current agent is the model's own previous pick, or with `given=True` the correct one, so one wrong label can't cascade.

In [11]:
def live_routing(model, convs, given=False):
    rows, lat = [], []
    for conv in convs:
        cur = None
        for u in conv["user_turns"]:
            history = conv["messages"][:u["message_index"] + 1]
            p, ms = timed_probs(model, routing_item(history, cur, conv["agents"]))
            pick = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv.get("company"), "type": u["turn_type"],
                         "flag": u.get("flag"), "ok": pick == u["expected_agent"]})
            lat.append(ms)
            cur = u["expected_agent"] if given else pick
    return rows, lat

def first_messages(model, msgs):
    rows, lat = [], []
    for m in msgs:
        p, ms = timed_probs(model, routing_item(user(m["message"]), None, m["agents"]))
        names, top = list(m["agents"]), p.topk(2).indices.tolist()
        rows.append({"type": m["type"], "ok": names[top[0]] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in top]})
        lat.append(ms)
    return rows, lat

def banking_routing(model):
    rows, lat = [], []
    for ex in banking:
        item = routing_item(ex["messages"], ex["current_agent"], BANKING_AGENTS)
        p, ms = timed_probs(model, item)
        rows.append({"ok": list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"]})
        lat.append(ms)
    return rows, lat

def questions(model, items):
    rows, lat = [], []
    for it in items:
        item = {"messages": user(it["state"]), "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        top = int(p.argmax())
        level_error = abs(top - it["answer_index"]) if it["type"] == "score" else None
        rows.append({**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"],
                     "conf": p.max().item(), "level_error": level_error})
        lat.append(ms)
    return rows, lat

Routing, first-message and banking tests fill `R = {result: (right, total)}` and `LAT = {test: latencies}`.

In [12]:
TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
right = {"right": lambda r: r["ok"]}

def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)

def test_routing(model, R, LAT):
    for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r)]:
        log(f"{name}: routing every turn")
        rows, LAT[f"{name} routing"] = live_routing(model, convs)
        given_rows = live_routing(model, convs, given=True)[0]
        R[f"{name} routing live"] = n_ok(rows)
        R[f"{name} routing given"] = n_ok(given_rows)
        if name == "E10":
            R["E10 test companies"] = n_ok([r for r in rows if r["company"] in TEST_COMPANIES])
        n_given = R[f"{name} routing given"][0]
        print(f"{name} routing live, per turn type (given previous agent: {n_given})")
        table(rows, lambda r: r["type"], right)

def test_first_messages(model, R, LAT):
    for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f)]:
        log(f"{name}: first messages")
        rows, LAT[f"{name} first"] = first_messages(model, msgs)
        R[f"{name} first top-1"] = n_ok(rows)
        R[f"{name} first top-2"] = n_ok(rows, "top2")
        table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})

def test_banking(model, R, LAT):
    log("banking")
    rows, LAT["banking"] = banking_routing(model)
    R["banking 50"] = n_ok(rows[:50])
    R["banking 225"] = n_ok(rows)

The question tests (with E12's mistakes and the independent set's flagged items), then `evaluate`: a warm-up question, every test, and the latencies.

In [13]:
def test_questions(model, R, LAT):
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        log(f"{name}: questions")
        rows, LAT[f"{name} questions"] = questions(model, items)
        R[f"{name} questions"] = n_ok(rows)
        for t in TYPE:
            of_type = [r for r in rows if r["type"] == t]
            if of_type:
                R[f"{name} {TYPE[t]}"] = n_ok(of_type)
        table(rows, lambda r: TYPE[r["type"]], right, [TYPE[t] for t in TYPE])
        sc = [r for r in rows if r["type"] == "score"]
        print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
        if name == "E12":
            table(rows, lambda r: r["category"], right)
            print("  mistakes:")
            for r in [r for r in rows if not r["ok"]]:
                print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | "
                      f"{r['question'][:36]:36s} | "
                      f"expected {r['answer']:14s} | got {r['pick']} ({r['conf']:.2f})")
        if name == "IND":
            R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])
            unflagged = R["IND questions unflagged"]
            claim_check = n_ok([r for r in rows if r.get("flag") == "claim_check"])[0]
            print(f"  without flagged items {unflagged[0]} / {unflagged[1]} | claim-check yes/no "
                  f"(truth always yes) {claim_check}")

def evaluate(model):
    model.train(False)
    timed_probs(model, routing_item(user("warm-up"), None, BANKING_AGENTS))
    R, LAT = {}, {}
    test_routing(model, R, LAT)
    test_first_messages(model, R, LAT)
    test_banking(model, R, LAT)
    test_questions(model, R, LAT)
    for name, L in LAT.items():
        L = lat_summary(L)
        print(f"latency {name:16s} p50 {L['p50']:4.0f} ms | p95 {L['p95']:4.0f} ms "
              f"({L['n']} questions)")
    return R, LAT

---
# Signpost: train and test

Up to 3 seeded attempts; a collapse moves to seed + 1000. Then every test, saved to `lora_signpost_results.json`. **Expect** ~30 minutes on a T4 and run 25's results within ±4: E10 78 / 83 and 181 / 193, E11 97, banking 48 · 210, E12 161, independent 221 · 133 · 180.

In [14]:
attempts = []
for attempt in range(ATTEMPTS):
    seed = SEED + SEED_STEP * attempt
    random.seed(seed)
    torch.manual_seed(seed)
    if AMP:
        torch.cuda.reset_peak_memory_stats(DEVICE)
    model = Model()
    head_m = sum(p.numel() for p in model.head.parameters()) / 1e6
    log(f"attempt {attempt + 1} (seed {seed}) | head {head_m:.2f} M | {gpu()}")
    model.enc.print_trainable_parameters()
    try:
        trained = train_model(model, seed)
        attempts.append({"seed": seed, "status": "ok"})
        break
    except Collapsed as e:
        log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
        attempts.append({"seed": seed, "status": f"collapsed: {e}"})
        model = None                        # free the failed attempt
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()
else:
    raise RuntimeError(f"training collapsed in all attempts: {attempts}")

R, LAT = evaluate(model)
peak_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
results = {"results": R, "latency": {k: lat_summary(v) for k, v in LAT.items()},
           "attempts": attempts, **trained, "peak_gpu_gb": peak_gb}
json.dump(results, open("lora_signpost_results.json", "w"), indent=1)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[06:17:08] attempt 1 (seed 42) | head 0.26 M | GPU 1.6 GB (peak 1.6)
trainable params: 7,196,672 || all params: 401,978,368 || trainable%: 1.7903
[06:17:08] training: 2180 batches per epoch (6319 routing + 9586 questions), 4360 steps, score loss ordinal
[06:17:15]   epoch 1 step 100/4360 | loss routing 1.5165 questions 0.9753 | 13.98 steps/s | ~5.1 min left | GPU 1.7 GB (peak 8.8)
[06:17:20]   epoch 1 step 200/4360 | loss routing 1.1525 questions 0.8759 | 16.30 steps/s | ~4.3 min left | GPU 1.7 GB (peak 8.9)
[06:17:25]   epoch 1 step 300/4360 | loss routing 0.6594 questions 0.6577 | 17.08 steps/s | ~4.0 min left | GPU 1.7 GB (peak 9.2)
[06:17:31]   epoch 1 step 400/4360 | loss routing 0.3097 questions 0.5820 | 17.47 steps/s | ~3.8 min left | GPU 1.7 GB (peak 9.6)
[06:17:36]   epoch 1 step 500/4360 | loss routing 0.3871 questions 0.5945 | 17.79 steps/s | ~3.6 min left | GPU 1.7 GB (peak 9.6)
[06:17:41]   epoch 1 step 600/4360 | loss routing 0.2411 questions 0.4109 | 17.95 steps/s | ~3.5

## Save and verify

`signpost/` gets the adapter, the merged fp16 encoder, `head.pt`, the config and `signpost.py`, zipped to `signpost.zip`. A reload then rescores E11 / E12: fp32 should match the trained model; fp16 can differ by an item (here E12 160 vs 159).

In [15]:
OUT = Path("signpost")
if SAVE_MODEL:
    shutil.rmtree(OUT, ignore_errors=True)
    model.enc.save_pretrained(OUT / "adapter")         # the LoRA adapter alone, before merging
    merged = model.enc.merge_and_unload()              # W <- W + (alpha / r) B A in all 112 layers
    merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
    model.tok.save_pretrained(OUT / "encoder")
    head_state = {k: v.float().cpu() for k, v in model.head.state_dict().items()}
    torch.save({"state_dict": head_state, "hidden": model.H}, OUT / "head.pt")
    cfg = {"name": "Signpost", "base_model": ENCODER, "license_of_base": "Apache-2.0", "sep": SEP,
           "prefix": "", "max_len": MAX_LEN, "route_question": ROUTE_Q, "lora_r": LORA_R,
           "lora_alpha": LORA_ALPHA, "lora": f"peft {peft.__version__}", "score_loss": SCORE_LOSS,
           "epochs": EPOCHS, "best_epoch": trained["best_epoch"], "seed_attempts": attempts,
           "trained_on": {"routing": len(train_route), "questions": len(train_choice)},
           "data_branch": DATA_BRANCH, "results": {k: v[0] for k, v in R.items()}}
    json.dump(cfg, open(OUT / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent.parent / "signpost" / "signpost.py", OUT / "signpost.py")
    shutil.make_archive("signpost", "zip", root_dir=".", base_dir=str(OUT))
    size = {d: sum(f.stat().st_size for f in (OUT / d).rglob("*") if f.is_file()) / 1e6
            for d in ("encoder", "adapter")}
    zip_gb = Path("signpost.zip").stat().st_size / 1e9
    log(f"saved {OUT}/ (encoder {size['encoder']:.0f} MB, adapter {size['adapter']:.0f} MB) "
        f"and signpost.zip ({zip_gb:.2f} GB)")

    sys.path.insert(0, str(OUT.resolve()))
    from signpost import Signpost
    for dtype in (torch.float16, torch.float32):       # fp16 = GPU use; fp32 = what a CPU computes
        sp = Signpost(OUT, device=DEVICE, dtype=dtype)
        n11 = sum(sp.route(user(m["message"]), m["agents"])[0] == m["target_agent"] for m in e11)
        n12 = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"]
                  for it in e12)
        log(f"reload check ({dtype}): E11 {n11}/{len(e11)} "
            f"(trained model {R['E11 first top-1'][0]}) | "
            f"E12 {n12}/{len(e12)} (trained model {R['E12 questions'][0]})")
        del sp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[06:21:57] saved signpost/ (encoder 793 MB, adapter 29 MB) and signpost.zip (0.76 GB)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[06:21:59] reload check (torch.float16): E11 96/100 (trained model 96) | E12 158/170 (trained model 158)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[06:22:01] reload check (torch.float32): E11 96/100 (trained model 96) | E12 158/170 (trained model 158)


---
# Results

This run next to run 25 (the published Signpost; its independent-2 numbers are from `05_listwise_signpost.ipynb`, branch `exp/listwise-index`), the frozen system (`02`, run 17) and the decider (`04`; no current-agent input, so no "given").

Each training run lands a little differently: the same recipe and seed differ by about ±4 items on most tests, and more on the independent questions, so compare the columns within that range.

In [16]:
REF = {
    "Signpost (run 25)": {
        "E10 test companies": 78, "E10 routing live": 181, "E11 first top-1": 97,
        "banking 50": 48, "banking 225": 210,
        "E12 questions": 161, "E12 yes/no": 58, "E12 choice": 58, "E12 score": 45,
        "IND questions": 221, "IND yes/no": 75, "IND choice": 103, "IND score": 43,
        "IND routing live": 133, "IND first top-1": 180,
        "IND2 questions": 188, "IND2 yes/no": 47, "IND2 choice": 47, "IND2 score": 45,
        "IND2 many-option": 49, "IND2 routing live": 62, "IND2 routing given": 65,
        "IND2 first top-1": 47},
    "frozen (02, run 17)": {
        "E10 test companies": 69, "E10 routing live": 158, "E11 first top-1": 94,
        "banking 50": 43, "banking 225": 194,
        "E12 questions": 114, "E12 yes/no": 38, "E12 choice": 44, "E12 score": 32},
    "decider 2B": {
        "E10 test companies": 62, "E10 routing live": 151, "E11 first top-1": 96,
        "banking 50": 48, "banking 225": 214,
        "E12 questions": 160, "E12 yes/no": 56, "E12 choice": 56, "E12 score": 48,
        "IND questions": 253, "IND yes/no": 77, "IND choice": 106, "IND score": 70,
        "IND questions unflagged": 253, "IND routing live": 113, "IND first top-1": 178,
        "IND2 questions": 185, "IND2 yes/no": 49, "IND2 choice": 46, "IND2 score": 43,
        "IND2 many-option": 47, "IND2 routing live": 55, "IND2 first top-1": 50},
}
print(f"{'result':24s} | {'this run':>11} | " + " | ".join(f"{c:>19s}" for c in REF))
for k, (ok_, n) in R.items():
    this_run = f"{ok_} / {n}"
    others = " | ".join(f"{REF[c].get(k, '—'):>19}" for c in REF)
    print(f"{k:24s} | {this_run:>11} | " + others)
print(f"\ntraining {trained['train_minutes']:.1f} min, best epoch {trained['best_epoch']} | "
      f"attempts {[a['status'] for a in attempts]} | peak GPU {peak_gb:.1f} GB")

result                   |    this run |   Signpost (run 25) | frozen (02, run 17) |          decider 2B
E10 routing live         |   177 / 193 |                 181 |                 158 |                 151
E10 routing given        |   179 / 193 |                   — |                   — |                   —
E10 test companies       |     77 / 83 |                  78 |                  69 |                  62
IND routing live         |   128 / 172 |                 133 |                   — |                 113
IND routing given        |   139 / 172 |                   — |                   — |                   —
IND2 routing live        |     61 / 70 |                  62 |                   — |                  55
IND2 routing given       |     61 / 70 |                  65 |                   — |                   —
E11 first top-1          |    96 / 100 |                  97 |                  94 |                  96
E11 first top-2          |   100 / 100 |               

---
## What we tried and dropped

Runs are in `experiments/modernbert_router_ledger.md`. Noise is about ±4 on most tests; independent questions vary more (232 and 219 / 300 on same-seed reruns).

| Run | Tried | Result | Decision |
|---|---|---|---|
| 18 | LoRA + head, questions only | E12 **114 → 153** / 170 (decider 160) | Kept: the plateau was the frozen encoder |
| 19 | One model for both jobs; ModernBERT-large vs Nemotron-1B | E10 test 75 and 79 vs decider 62 / 83; E12 153 vs 141 | One model; ModernBERT reads better |
| 20 | More score data (709 → 2,004 items) | E12 score stayed 43 / 53 | Dropped: taught a shortcut |
| 21 | Cleaned data, then plain vs ordinal score loss | Plain E12 155; ordinal **161**, score 47 | Ordinal adopted |
| 22 | Same recipe, rerun | Collapsed: E12 88, dev choice 599 / 1,014 | Collapse guard; rerun 22b healthy (E12 161) |
| 23 | Nemotron-1B, final recipe | Routing tied (134 vs 131); questions worse (200 vs 231) | ModernBERT-large stays |
| 24 | Two specialists vs one model | No specialist better beyond noise | One model |
| after 25 | Hand-written score data | No gain (135 vs 133 / 150) | Dropped |

Also dropped: `02`'s detector and stay gate. Independent-set limits (never fed back; on the model card): routing holds up (133–135 vs 113 / 172), score precision does not (42–53 vs 70 / 92), and stay turns switch too readily (42 vs 49 / 55). Since run 25, LoRA comes from `peft`.

## Next

**Part 5** (branch `signpost-apps`) uses Signpost as the router of a LangGraph app with five Claude agents.